## imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, KFold, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# regressors
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR

# classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")

## 1. data audit

In [ ]:
# search kaggle path dynamically
data_path = None
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".csv") and "coaster" in file.lower():
            data_path = os.path.join(root, file)
            break

if data_path is None:
    data_path = "coaster_db.csv"

df = pd.read_csv(data_path)
print(f"loaded dataset from: {data_path}")

# standardizing column names
col_map = {col.lower(): col for col in df.columns}
speed_col = col_map.get('speed_mph', 'speed_mph')
height_col = col_map.get('height_ft', 'height_ft')
length_col = col_map.get('length_ft', 'Length_ft' if 'Length_ft' in df.columns else 'length_ft')
inversions_col = col_map.get('inversions_clean', 'Inversions_clean' if 'Inversions_clean' in df.columns else 'inversions_clean')
gforce_col = col_map.get('gforce_clean', 'Gforce_clean' if 'Gforce_clean' in df.columns else 'gforce_clean')
type_col = col_map.get('type_main', 'Type_Main' if 'Type_Main' in df.columns else 'type_main')

print("\n--- shape ---")
print(df.shape)

print("\n--- dtypes ---")
print(df.dtypes)

print("\n--- missing values ---")
print(df.isnull().sum())

print("\n--- duplicates ---")
print(df.duplicated().sum())

print(f"\n--- target stats ({speed_col}) ---")
print(df[speed_col].describe())

## 2. EDA

In [ ]:
# target distribution
plt.figure(figsize=(8, 4))
sns.histplot(df[speed_col].dropna(), kde=True, color="teal")
plt.title(f"target distribution: {speed_col}")
plt.xlabel("speed (mph)")
plt.ylabel("count")
plt.tight_layout()
plt.show()